In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat


pd.set_option("display.max_columns", 30)

In [ ]:
# ============================================================================
# TASK 1: Define the data root path
# ============================================================================
# Complete the ROOT_PATH by navigating to the data directory from BASE_DIR

BASE_DIR = Path().resolve()
# Define data root path based on the current file location
ROOT_PATH = BASE_DIR / ".." / ".." / ".." / "data" / "0_raw" / "tanzania"
DATASET = 'survey_ces'
DATA_PATH = ROOT_PATH / DATASET

In [ ]:
# ============================================================================
# TASK 2: Load the data files
# ============================================================================
# We load three related datasets from the CES survey. If encoding issues arise,
# try different encodings (utf8, latin1) as a parameter to read_dta()

df_ces,   meta_ces   = pyreadstat.read_dta(DATA_PATH / "CES.dta")
df_sec1R, meta_sec1R = pyreadstat.read_dta(DATA_PATH / "sec1R.dta")
df_sec20, meta_sec20 = pyreadstat.read_dta(DATA_PATH / "sec20R1.dta")

print(df_ces.shape, df_sec1R.shape, df_sec20.shape)

In [ ]:
ces_keep_labels = [
    # Identifiers, status, completion time
    "Interview key (identifier in XX-XX-XX-XX format)",
    "Unique 32-character long identifier of the interview",
    "Establishment Unique Number",
    "Status of the Interview",
    "datetime: last interview completion",

    # Geography
    "Region",
    "District",
    "Council",

    # Sector and legal form
    "Main Previous  ISIC",
    "What is this firm’s current legal status",
    "Screening1"

    # Firm timeline
    "In what year did this establishment begin operations?",
    "Establishment Age",

    # Ownership and gender
    "Amongst the owners of the firm, are there any females?",
    "What percentage is owned by females?",

    # Workforce
    "All Workers (s3q1a and s3q21 s3q22)",

    # Sales and exports
    "In 2025, what were total annual sales for this establishment (turnover)?",
    "Do you Export?",
    "National sales",
    "Indirect exports",
    "Direct exports",

    # Obstacle ratings (1–5; 6 = does not apply)
    "Labor regulations",
    "Inadequately educated workforce",
    "Using the response options provided ; To what degree is electricity an obstacle",
    "To what degree is access to land an obstacle to the current operations of this e",
    "To what degree are practices of competitors in the informal sector an obstacle t",
    "To what degree is access to finance an obstacle to the current operations of thi",
]

sec1R_keep_labels = [
    "Unique 32-character long identifier of the interview",
    "Id in sec1R",
    "Percentage of ownership",
]

sec20_keep_labels = [
    "Unique 32-character long identifier of the interview",
    "Id in sec20R1",
    "Income Statement for year 2025 in TZS",
]

In [ ]:
def labels_to_cols(labels, meta):
    rev = {lab: col for col, lab in meta.column_names_to_labels.items()}
    return [rev[l] for l in labels]

# BEFORE PROCEEDING CHECK THE FUNCTION ABOVE. IT HAS A BUG. WHAT IS IT? THE FUNCTION BELOW SOLVES IT, CHECK IT OUT ONCE YOU FIND THE BUG

In [ ]:
def labels_to_cols(labels, meta):
    column_names = [col for col, lab in meta.column_names_to_labels.items() if lab in labels]
    return column_names

In [ ]:

df_ces   = df_ces[labels_to_cols(ces_keep_labels,   meta_ces)].copy()
df_sec1R = df_sec1R[labels_to_cols(sec1R_keep_labels, meta_sec1R)].copy()
df_sec20 = df_sec20[labels_to_cols(sec20_keep_labels, meta_sec20)].copy()

print(df_ces.columns.tolist())

In [ ]:
meta_ces.column_names_to_labels['s1q15a']

In [ ]:
# ============================================================================
# TASK 3: Explore the datasets
# ============================================================================
# Write commands to get a first understanding of the datasets:
# - Get basic info (shape, dtypes, head)
# - Take a sample of rows
# - Transpose a subset to see columns as rows (useful for wide datasets)
# - Any other exploratory commands you find useful

# Write your exploration code here
print("CES Dataset:")
print(df_ces.shape)
print(df_ces.dtypes)
print(df_ces.head())
print("\nSec1R Dataset:")
print(df_sec1R.shape)
print(df_sec1R.dtypes)
print(df_sec1R.head())
print("\nSec20 Dataset:")
print(df_sec20.shape)
print(df_sec20.dtypes)
print(df_sec20.head())

In [ ]:
# ============================================================================
# TASK 4: Keep only completed interviews
# ============================================================================
# Filter the dataset to include only interviews with a "completed" status.
# Use the "Status of the Interview" column to identify completed interviews.

print("Before:", df_ces.shape)

interview_status_col = next(key for key, label in meta_ces.column_names_to_labels.items() if label == "Status of the Interview")

print(f"Interview status column: {interview_status_col}")

print(meta_ces.variable_value_labels[interview_status_col])

# Write filtering code here to keep only completed interviews
df_ces = df_ces[df_ces[interview_status_col] == 1]  # Keep only completed interviews (status == 1)

print("After:", df_ces.shape)

In [ ]:
# ============================================================================
# TASK 5: Fix data types
# ============================================================================
# Convert columns to appropriate data types for analysis.
# This ensures correct operations and reduces memory usage.

df_ces["interview__id"] = df_ces["interview__id"].astype(str)
df_ces["AssNO"]         = df_ces["AssNO"].astype(str)

# Convert tmlcmp to datetime, then extract year and month from it
df_ces["tmlcmp"]          = pd.to_datetime(df_ces["tmlcmp"])
df_ces["interview_year"]  = df_ces["tmlcmp"].dt.year
df_ces["interview_month"] = df_ces["tmlcmp"].dt.month

# Convert region, s1q15a, s2q12 to category type for efficient storage and operations
for col in ["region", "s1q15a", "s2q12"]:
    df_ces[col] = df_ces[col].astype("category")

In [ ]:
# ============================================================================
# TASK 6: Remove duplicate establishments, keeping the most recent interview
# ============================================================================
# Some establishments appear multiple times (re-interviewed). Keep only the
# most recent interview for each establishment (identified by AssNO), based on tmlcmp.

print("Duplicate AssNO:", df_ces.duplicated(subset=["AssNO"]).sum())

# Sort by AssNO and tmlcmp, then drop duplicates keeping the last (most recent)
df_ces = df_ces.sort_values("tmlcmp").drop_duplicates(subset=["AssNO"], keep="last")

print("After:", df_ces.shape)

In [ ]:
# ============================================================================
# TASK 7: Map numeric values to readable labels
# ============================================================================
# Convert numeric codes to human-readable labels using the metadata.

region_map       = meta_ces.variable_value_labels["region"]
sector_map       = meta_ces.variable_value_labels["s1q15a"]
legal_status_map = meta_ces.variable_value_labels["s2q12"]

# Use .map() to create new columns with readable labels
df_ces["region_name"]       = df_ces["region"].map(region_map)
df_ces["sector_name"]       = df_ces["s1q15a"].map(sector_map)
df_ces["legal_status_name"] = df_ces["s2q12"].map(legal_status_map)

In [ ]:
df_ces["sector_name"].value_counts().plot(kind="barh", title="Firms by sector")

In [ ]:
# ============================================================================
# TASK 8: Recode invalid values to NaN
# ============================================================================
# In survey data, certain values (like 6) indicate "does not apply" or are invalid.
# Replace these with np.nan so they're excluded from analysis.
# Use .where() to keep values where a condition is True, replacing others with NaN.

severity_cols = ["s9q11", "s12q10", "s15q8", "s16q33", "s3q33a", "s3q33b"]

# STUDENT: Use .loc or .iloc to keep values != 6, replacing 6 with np.nan in each severity column
for col in severity_cols:
    ...

# STUDENT: Use .where() to set non-positive s7q26 (sales) values to np.nan
df_ces["s7q26"] = ...

In [ ]:
# ============================================================================
# TASK 9: Derive new analytical columns
# ============================================================================
# Create new features that will be useful for analysis:
# - firm_age: how old is the establishment
# - is_female_owned: whether firm has female ownership
# - is_exporter: whether firm participates in exports
# - size_band: categorize firms by workforce size

df_ces = df_ces.assign(
    # Define firm age based on current year (2026)
    firm_age        = ...,
    # Define if a company is female owned (check ownership percentage column)
    is_female_owned =  ...,
    # True if direct-exports share > 10, find the columns for that
    is_exporter     = ...,
    # Bin "Workers" with bins [0, 5, 19, 99, np.inf] and labels Micro/Small/Medium/Large
    size_band       = ...,
)

In [ ]:
df_ces["firm_age"].plot.hist(bins=30, title="Firm age distribution")

In [ ]:
# ============================================================================
# TASK 10: Explore the ownership roster data
# ============================================================================
# The sec1R dataset is in "long" format with multiple ownership rows per company.
# Let's understand the structure before reshaping it to join with the main table.

df_sec1R.head()

df_sec1R.shape, df_sec1R["sec1R__id"].value_counts().sort_index()

owner_labels = meta_sec1R.variable_value_labels["sec1R__id"]

# Attach a readable label per row using .apply() + lambda
df_sec1R["owner_type"] = df_sec1R["sec1R__id"].apply(lambda x: )

# Check how many rows per company (expect 4 on average, one per owner type)
df_sec1R.groupby ...

In [ ]:

df_sec1R["owner_type"].value_counts().plot(kind="barh", title="Roster rows per owner type")

In [ ]:
# ============================================================================
# TASK 11: Transform ownership data to wide format for merging
# ============================================================================
# Why wide format? Each company should be ONE row in the master table (df_ces).
# By pivoting ownership data, we create one column per owner type, making it easy to join.

# Use pivot_table with interview__id as rows and owner type info as columns
ownership_wide = df_sec1R.pivot_table(
    index=...,
    columns=...,
    values=...,
    aggfunc="first",
).reset_index()

ownership_wide = ownership_wide.rename(columns={
    owner_labels[1]: "own_private_domestic",
    owner_labels[2]: "own_private_foreign",
    owner_labels[3]: "own_government",
    owner_labels[4]: "own_other",
})

ownership_wide.head()

In [ ]:
# Merge the pivoted ownership data onto the main dataset using left join on interview__id
df_ces = ...

print(df_ces.shape)

In [ ]:
# ============================================================================
# TASK 12: Explore the income statement data
# ============================================================================
# Similar to the ownership data, income statements have multiple rows per company
# (one per line item). We'll explore, filter, and reshape for merging.

df_sec20.head()

In [ ]:
item_labels = meta_sec20.variable_value_labels["sec20R1__id"]

# Create a truncated label (45 chars) for better visualization in the plot
counts = df_sec20["sec20R1__id"].value_counts().sort_index()
counts.index = counts.index.map( ... )
counts.plot(kind="barh", figsize=(8, 6), title="Roster rows per income-statement item")

In [ ]:
# ============================================================================
# TASK 13: Filter income statement to key items
# ============================================================================
# We only need labour force costs and raw materials for this analysis.
# Find the correct codes and filter the data.

# Filter to keep only labour force costs and raw materials (check variable values to find codes)
df_sec20_subset =  # filter here
# Map the item IDs to readable names using the item_labels dictionary
df_sec20_subset["item_name"] = df_sec20_subset["sec20R1__id"].map(... )
df_sec20_subset.head()

In [ ]:
# ============================================================================
# TASK 14: Reshape income data to wide format for merging
# ============================================================================
# Reshape income to wide format so we have one row per company with income items as columns.
# Note: pivot_table aggregates duplicates; pivot doesn't and raises if any exist.
# Since (interview__id, item_name) is unique here, pivot is the cleaner choice.

# Use pivot to reshape from long to wide format (index, columns, values)
income_wide = df_sec20_subset.pivot(
    index=...,
    columns=...,
    values=...,
).reset_index()

income_wide = income_wide.rename(columns={
    item_labels[1]:  "labor_cost_2025",
    item_labels[13]: "raw_materials_2025",
})

income_wide.head()

In [ ]:
# ============================================================================
# TASK 15: Merge income data into master table
# ============================================================================
# Join the income data onto the main df_ces table using the interview__id.

# Left-merge income_wide onto df_ces
df_ces = ...

print(df_ces.shape)

In [ ]:
# ============================================================================
# SUMMARY: Verify data quality and prepare for analysis
# ============================================================================
# Review final dataset structure, missing values, and summary statistics
# to ensure the data is ready for analysis.

print("Final shape:", df_ces.shape)
print("\nDtype distribution:")
print(df_ces.dtypes.value_counts())

quality_cols = [
    "firm_age", "Workers", "s7q26", "labor_cost_2025",
    "own_private_domestic", "own_private_foreign",
    "own_government", "own_other",
]
print("\nMissing % for key analytical columns:")
print((df_ces[quality_cols].isna().mean() * 100).round(1))

print("\nMedian total sales (TZS) by sector and size band:")
print(
    df_ces.groupby(["sector_name", "size_band"], observed=True)["s7q26"]
          .median()
          .unstack()
          .round(0)
)

In [ ]:
# ============================================================================
# TASK 16: Save the cleaned dataset
# ============================================================================
# Define output directory and save the processed dataset for future analysis.

# Define output directory path
OUT_DIR =
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Save the dataset (choose format: csv, parquet, etc.)
df_ces.
print("Saved.")